# RAG Pipeline Evaluation — Agents Demo

This notebook demonstrates how to evaluate the RAG (Retrieval-Augmented Generation) pipeline used by the **Atlas** and **Arquimedes** agents.

## Evaluation Metrics
- **Context Relevance**: Does the retrieved context answer the question?
- **Answer Faithfulness**: Is the generated answer grounded in the retrieved context?
- **Chunk Quality**: Are the text chunks the right size for retrieval?

---

In [ ]:
# ─── Dependencies ─────────────────────────────────────────────────────────────
import json
import re
from typing import List, Dict, Any, Optional
from dataclasses import dataclass, field
from pathlib import Path

print('✅ Dependencies loaded')

## 1. Chunking Strategy Analysis

One of the most critical decisions in a RAG pipeline is how to split documents into chunks. We compare three strategies:
- **Fixed-size**: Split by character count
- **Sentence-based**: Split by sentence boundaries
- **Semantic**: Split by topic/section

In [ ]:
@dataclass
class Chunk:
    """Represents a text chunk from a document."""
    text: str
    chunk_id: int
    source: str
    token_count: int = 0
    
    def __post_init__(self):
        # Approximate token count (1 token ≈ 4 chars)
        self.token_count = len(self.text) // 4


def fixed_size_chunker(text: str, chunk_size: int = 512, overlap: int = 64) -> List[Chunk]:
    """Split text into fixed-size chunks with overlap."""
    chunks = []
    start = 0
    chunk_id = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk_text = text[start:end].strip()
        if chunk_text:
            chunks.append(Chunk(text=chunk_text, chunk_id=chunk_id, source='fixed'))
            chunk_id += 1
        start += chunk_size - overlap
    return chunks


def sentence_chunker(text: str, max_sentences: int = 5) -> List[Chunk]:
    """Split text by sentences, grouping N sentences per chunk."""
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks = []
    for i in range(0, len(sentences), max_sentences):
        group = sentences[i:i + max_sentences]
        chunk_text = ' '.join(group).strip()
        if chunk_text:
            chunks.append(Chunk(text=chunk_text, chunk_id=i // max_sentences, source='sentence'))
    return chunks


# Test with a sample document
sample_doc = """
LangGraph is a library for building stateful, multi-actor applications with LLMs.
It extends LangChain with the ability to coordinate multiple agents in a graph structure.
Each node in the graph represents an agent or a processing step.
Edges define the flow of information between nodes.
The state is shared across all nodes and persists throughout the execution.
This makes it ideal for complex workflows that require memory and coordination.
RAG (Retrieval-Augmented Generation) can be integrated as a node in the graph.
The retrieval node fetches relevant documents from a vector store.
The generation node uses the retrieved context to produce a grounded answer.
This architecture reduces hallucinations and improves factual accuracy.
"""

fixed_chunks = fixed_size_chunker(sample_doc, chunk_size=200, overlap=30)
sentence_chunks = sentence_chunker(sample_doc, max_sentences=3)

print(f'Fixed-size chunking: {len(fixed_chunks)} chunks')
print(f'Sentence-based chunking: {len(sentence_chunks)} chunks')
print(f'\nSample fixed chunk (tokens ≈ {fixed_chunks[0].token_count}):')
print(f'  "{fixed_chunks[0].text[:100]}..."')

## 2. Retrieval Quality Evaluation

We simulate a retrieval evaluation using cosine similarity between query and chunk embeddings (mocked for demo purposes).

In [ ]:
import math
import random

random.seed(42)


def mock_embed(text: str) -> List[float]:
    """Mock embedding — in production, use OpenAI/Anthropic/HuggingFace embeddings."""
    # Deterministic mock based on text hash
    random.seed(hash(text) % (2**32))
    vec = [random.gauss(0, 1) for _ in range(128)]
    norm = math.sqrt(sum(x**2 for x in vec))
    return [x / norm for x in vec]


def cosine_similarity(a: List[float], b: List[float]) -> float:
    """Compute cosine similarity between two vectors."""
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x**2 for x in a))
    norm_b = math.sqrt(sum(x**2 for x in b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)


def retrieve_top_k(query: str, chunks: List[Chunk], k: int = 3) -> List[Dict]:
    """Retrieve top-k most relevant chunks for a query."""
    query_emb = mock_embed(query)
    scored = []
    for chunk in chunks:
        chunk_emb = mock_embed(chunk.text)
        score = cosine_similarity(query_emb, chunk_emb)
        scored.append({'chunk': chunk, 'score': score})
    scored.sort(key=lambda x: x['score'], reverse=True)
    return scored[:k]


# Evaluate retrieval
test_queries = [
    'How does LangGraph handle state between agents?',
    'What is the role of RAG in reducing hallucinations?',
    'How are edges defined in a LangGraph workflow?',
]

print('=== Retrieval Evaluation ===')
for query in test_queries:
    results = retrieve_top_k(query, sentence_chunks, k=2)
    print(f'\nQuery: "{query}"')
    for i, r in enumerate(results):
        print(f'  [{i+1}] Score: {r["score"]:.4f} | "{r["chunk"].text[:80]}..."')

## 3. Answer Faithfulness Scoring

We evaluate whether the generated answer is grounded in the retrieved context using keyword overlap as a proxy metric.

In [ ]:
def faithfulness_score(answer: str, context_chunks: List[str]) -> float:
    """
    Compute a simple faithfulness score based on keyword overlap.
    In production, use an LLM judge (e.g., Claude or GPT-4o) for this.
    """
    answer_words = set(re.findall(r'\b\w{4,}\b', answer.lower()))
    context_text = ' '.join(context_chunks).lower()
    context_words = set(re.findall(r'\b\w{4,}\b', context_text))
    
    if not answer_words:
        return 0.0
    
    overlap = answer_words & context_words
    return len(overlap) / len(answer_words)


# Simulate QA evaluation
qa_pairs = [
    {
        'question': 'How does LangGraph manage state?',
        'answer': 'LangGraph uses a shared state that persists across all nodes throughout execution.',
        'context': [c.text for c in sentence_chunks[:3]]
    },
    {
        'question': 'What is RAG?',
        'answer': 'RAG stands for Retrieval-Augmented Generation. It fetches relevant documents and uses them to generate grounded answers.',
        'context': [c.text for c in sentence_chunks[2:]]
    },
    {
        'question': 'What is the capital of France?',  # Out-of-context hallucination test
        'answer': 'The capital of France is Paris, which is located in Western Europe.',
        'context': [c.text for c in sentence_chunks[:2]]
    },
]

print('=== Faithfulness Evaluation ===')
scores = []
for pair in qa_pairs:
    score = faithfulness_score(pair['answer'], pair['context'])
    scores.append(score)
    status = '✅ Grounded' if score > 0.4 else '⚠️ Potential hallucination'
    print(f'\nQ: {pair["question"]}')
    print(f'Faithfulness: {score:.2%} — {status}')

avg_faithfulness = sum(scores) / len(scores)
print(f'\n📊 Average Faithfulness Score: {avg_faithfulness:.2%}')

## 4. Summary & Recommendations

Based on this evaluation:

In [ ]:
summary = {
    'chunking_strategy': 'sentence-based (3 sentences/chunk)',
    'avg_chunk_tokens': sum(c.token_count for c in sentence_chunks) / len(sentence_chunks),
    'total_chunks': len(sentence_chunks),
    'avg_faithfulness': avg_faithfulness,
    'recommendations': [
        'Use sentence-based chunking for better semantic coherence',
        'Add an LLM-as-judge step for production faithfulness evaluation',
        'Consider pgvector or ChromaDB for scalable vector storage',
        'Implement hybrid search (BM25 + semantic) for better recall',
    ]
}

print('=== RAG Pipeline Summary ===')
print(f"Chunking Strategy : {summary['chunking_strategy']}")
print(f"Avg Chunk Tokens  : {summary['avg_chunk_tokens']:.1f}")
print(f"Total Chunks      : {summary['total_chunks']}")
print(f"Avg Faithfulness  : {summary['avg_faithfulness']:.2%}")
print('\n📋 Recommendations:')
for i, rec in enumerate(summary['recommendations'], 1):
    print(f'  {i}. {rec}')